# Decisiones base - Asistente de consulta normativa laboral

Notebook para sentar las bases antes del fine-tuning del cross-encoder: elegir modelo base (comparando tokenización sobre un PDF real del dominio), validar el formato del dataset de pares, y correr un sanity check zero-shot como referencia del baseline.

**Orden:**
1. Setup
2. Candidatos de modelo base
3. Comparación de tokenización sobre un PDF real del dominio
4. Decisión razonada
5. Esqueleto del dataset (positivo / negativo fácil / negativo difícil)
6. Sanity check zero-shot


## 1. Setup

In [22]:
# En Colab: descomenta la siguiente línea (o usa `pip install -r requirements.txt` en el venv local)
# !pip install -q transformers torch pandas pypdf

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
else:
    print("No hay GPU - corriendo en CPU (más lento, pero este notebook solo carga tokenizers y hace un sanity check, así que funciona bien)")


GPU disponible: False
No hay GPU - corriendo en CPU (más lento, pero este notebook solo carga tokenizers y hace un sanity check, así que funciona bien)


## 2. Candidatos de modelo base

Tres candidatos a comparar:
- `dccuchile/bert-base-spanish-wwm-cased` (BETO) - BERT monolingüe en español, entrenado sobre un corpus grande de texto en español (Whole Word Masking).
- `bert-base-multilingual-cased` (mBERT) - BERT multilingüe, 104 idiomas, vocabulario compartido.
- `distilbert-base-multilingual-cased` - versión liviana de mBERT, como referencia de tamaño/costo.

Cambien o agreguen candidatos aquí si quieren comparar otros.

In [23]:
MODEL_CANDIDATES = {
    "beto-spanish": "dccuchile/bert-base-spanish-wwm-cased",
    "mbert": "bert-base-multilingual-cased",
    "distilbert-multilingual": "distilbert-base-multilingual-cased",
}

tokenizers = {}
for name, checkpoint in MODEL_CANDIDATES.items():
    print(f"Cargando tokenizer: {name} ({checkpoint})")
    tokenizers[name] = AutoTokenizer.from_pretrained(checkpoint)

print("\nListo. Tokenizers cargados:", list(tokenizers.keys()))


Cargando tokenizer: beto-spanish (dccuchile/bert-base-spanish-wwm-cased)
Cargando tokenizer: mbert (bert-base-multilingual-cased)
Cargando tokenizer: distilbert-multilingual (distilbert-base-multilingual-cased)

Listo. Tokenizers cargados: ['beto-spanish', 'mbert', 'distilbert-multilingual']


## 3. Comparación de tokenización sobre un PDF real del dominio

Se corre la tokenización sobre texto real y largo (un PDF del dominio), y se mide la **fertilidad** (tokens por palabra) sobre el documento completo. Esto es más representativo que evaluar palabras sueltas: captura la variación real de vocabulario del dominio.

Coloca el PDF en `data/SL2600-2025.pdf` y ajusta `PDF_PATH` en la siguiente celda si el nombre de archivo es distinto. Ese archivo puntual sí se versiona (es una sentencia pública, no datos sensibles) para que el notebook sea reproducible sin pasos manuales - el resto de `data/` sigue ignorado. Se extrae el texto completo y se mide, para cada tokenizer:
- **Total de tokens** que produce sobre el documento (relevante para costo/ventana de contexto).
- **Fertilidad** (tokens ÷ palabras): en promedio, ¿en cuántos tokens se parte cada palabra? Menor es mejor - significa que el tokenizer trata el vocabulario del dominio como unidades más completas.

In [24]:
from pathlib import Path

PDF_PATH = Path("../data/SL2600-2025.pdf")  # ajusta el nombre si tu PDF es otro

try:
    from google.colab import files
    uploaded = files.upload()  # selecciona el PDF cuando aparezca el diálogo
    pdf_filename = list(uploaded.keys())[0]
except ImportError:
    if not PDF_PATH.exists():
        raise FileNotFoundError(
            f"No se encontró {PDF_PATH}. Coloca el PDF del dominio en esa ruta "
            "o ajusta PDF_PATH arriba."
        )
    pdf_filename = str(PDF_PATH)

print("Archivo cargado:", pdf_filename)


Archivo cargado: ..\data\SL2600-2025.pdf


In [25]:
from pypdf import PdfReader

reader = PdfReader(pdf_filename)
print(f"Páginas: {len(reader.pages)}")

full_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + " "

full_text = " ".join(full_text.split())  # normaliza espacios/saltos de línea
print(f"Caracteres extraídos: {len(full_text)}")
print(f"Primeros 50 caracteres:\n{full_text[:100]}")


Páginas: 39
Caracteres extraídos: 62439
Primeros 50 caracteres:
SCLAJPT-10 V.00 LUIS BENEDICTO HERRERA DÍAZ Magistrado ponente SL2600-2025 Radicación n.o 11001-31-0


In [26]:
import pandas as pd

n_words = len(full_text.split())
print(f"Palabras (aprox., split por espacio): {n_words}\n")

rows = []
for name, tok in tokenizers.items():
    n_tokens = len(tok.tokenize(full_text))
    fertility = n_tokens / n_words
    rows.append({
        "modelo": name,
        "total_tokens": n_tokens,
        "total_palabras": n_words,
        "fertilidad (tokens/palabra)": round(fertility, 3),
    })

df_pdf_comparison = pd.DataFrame(rows).sort_values("fertilidad (tokens/palabra)")
df_pdf_comparison


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (13518 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (15677 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (15677 > 512). Running this sequence through the model will result in indexing errors


Palabras (aprox., split por espacio): 10031



,modelo,total_tokens,total_palabras,fertilidad (tokens/palabra)
0,beto-spanish,13518,10031,1.348
1,mbert,15677,10031,1.563
2,distilbert-multilingual,15677,10031,1.563


In [27]:
# Ver ejemplos concretos de palabras del PDF que cada tokenizer parte en más pedazos
import re
from collections import Counter

# palabras únicas del documento, filtrando cortas y muy comunes
words = [w for w in re.findall(r"[a-záéíóúñA-ZÁÉÍÓÚÑ]{6,}", full_text)]
sample_words = [w for w, _ in Counter(words).most_common(10)]

print("Palabras frecuentes del documento y cómo las parte cada tokenizer:\n")
for w in sample_words:
    partes = {name: tok.tokenize(w) for name, tok in tokenizers.items()}
    print(f"{w}:")
    for name, toks in partes.items():
        print(f"   {name}: {toks}")
    print()


Palabras frecuentes del documento y cómo las parte cada tokenizer:

laboral:
   beto-spanish: ['laboral']
   mbert: ['labor', '##al']
   distilbert-multilingual: ['labor', '##al']

pensión:
   beto-spanish: ['pensión']
   mbert: ['pen', '##sión']
   distilbert-multilingual: ['pen', '##sión']

trabajador:
   beto-spanish: ['trabajador']
   mbert: ['trabaja', '##dor']
   distilbert-multilingual: ['trabaja', '##dor']

derecho:
   beto-spanish: ['derecho']
   mbert: ['derecho']
   distilbert-multilingual: ['derecho']

SCLAJPT:
   beto-spanish: ['SC', '##LA', '##J', '##P', '##T']
   mbert: ['SC', '##LA', '##J', '##P', '##T']
   distilbert-multilingual: ['SC', '##LA', '##J', '##P', '##T']

Radicación:
   beto-spanish: ['Rad', '##icación']
   mbert: ['Rad', '##ica', '##ción']
   distilbert-multilingual: ['Rad', '##ica', '##ción']

estabilidad:
   beto-spanish: ['estabilidad']
   mbert: ['esta', '##bilidad']
   distilbert-multilingual: ['esta', '##bilidad']

artículo:
   beto-spanish: ['artícu

## 4. Decisión razonada

**Modelo base elegido:** `beto-spanish` (`dccuchile/bert-base-spanish-wwm-cased`)

**Por qué:** sobre el PDF de prueba (39 páginas, 10.031 palabras), la comparación de fertilidad (tokens/palabra) dio:

* beto-spanish: 1.348
* mbert: 1.563
* distilbert-multilingual: 1.563

BETO produce ~2.159 tokens menos que mBERT/DistilBERT sobre el mismo texto (13.518 vs. 15.677), con una fertilidad de 1.348 tokens por palabra frente a 1.563 de los multilingües. Es decir, BETO trata el vocabulario del dominio como unidades más completas, mientras que los multilingües lo fragmentan más - consistente con que su vocabulario compartido entre 104 idiomas deja menos espacio para vocabulario específico del español legal.

**Trade-offs considerados:** mBERT y DistilBERT-multilingual dieron exactamente la misma fertilidad porque comparten el mismo tokenizer (DistilBERT reutiliza el vocabulario de BERT multilingüe, solo cambia la arquitectura del modelo, no la tokenización). BETO es monolingüe, así que se descarta la ventaja de mBERT de cubrir otros idiomas - pero el proyecto es 100% en español, así que esa ventaja no aplica aquí. Los tres candidatos corren sin problema en el T4 gratis de Colab, así que el tamaño no fue un factor decisivo.


## 5. Esqueleto del dataset (positivo / negativo fácil / negativo difícil)

Estructura de ejemplo para validar el formato antes de generar el dataset real con el LLM sintetizador. Cada fila es un par `(consulta, artículo, etiqueta)`.

In [28]:
example_pairs = [
    {
        "consulta": "me despidieron sin avisar, ¿me deben algo?",
        "articulo": "CST Art. 64 - indemnización por terminación sin justa causa",
        "tipo": "positivo",
        "label": 1,
    },
    {
        "consulta": "me despidieron sin avisar, ¿me deben algo?",
        "articulo": "CST Art. 161 - jornada máxima legal",
        "tipo": "negativo_facil",
        "label": 0,
    },
    {
        "consulta": "me despidieron sin avisar, ¿me deben algo?",
        "articulo": "CST Art. 62 - justas causas de terminación del contrato",
        "tipo": "negativo_dificil",
        "label": 0,
    },
]

df_example = pd.DataFrame(example_pairs)
df_example


,consulta,articulo,tipo,label
0,"me despidieron sin avisar, ¿me deben algo?",CST Art. 64 - indemnización por terminación si...,positivo,1
1,"me despidieron sin avisar, ¿me deben algo?",CST Art. 161 - jornada máxima legal,negativo_facil,0
2,"me despidieron sin avisar, ¿me deben algo?",CST Art. 62 - justas causas de terminación del...,negativo_dificil,0


## 6. Sanity check zero-shot

Cargar el modelo elegido (sección 4) sin fine-tuning y correrlo sobre un par de ejemplo, para confirmar que el pipeline completo funciona de punta a punta antes de meterse al entrenamiento. Los scores acá **no van a ser buenos** - el modelo todavía no aprendió la tarea. Esto es solo para validar la mecánica y, más adelante, sirve como referencia del baseline zero-shot que pide la rúbrica.

In [29]:
CHOSEN_MODEL = MODEL_CANDIDATES["beto-spanish"]

tokenizer = AutoTokenizer.from_pretrained(CHOSEN_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(CHOSEN_MODEL, num_labels=2)
model.eval()
if torch.cuda.is_available():
    model = model.to("cuda")

print(f"Modelo cargado: {CHOSEN_MODEL}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 27423.76it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from differe

Modelo cargado: dccuchile/bert-base-spanish-wwm-cased


In [30]:
def score_pair(consulta, articulo):
    inputs = tokenizer(consulta, articulo, return_tensors="pt", truncation=True)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    return probs[0, 1].item()  # probabilidad de la clase "relevante"

for _, row in df_example.iterrows():
    score = score_pair(row["consulta"], row["articulo"])
    print(f"[{row['tipo']}] score={score:.4f}  articulo={row['articulo']}")

[positivo] score=0.5198  articulo=CST Art. 64 - indemnización por terminación sin justa causa
[negativo_facil] score=0.5449  articulo=CST Art. 161 - jornada máxima legal
[negativo_dificil] score=0.5372  articulo=CST Art. 62 - justas causas de terminación del contrato
